# Intro to Machine Learning

Notebook ini membahas konsep dasar Machine Learning menggunakan dataset properti Melbourne.
Kita akan membangun model **Decision Tree** dan mencari ukuran pohon terbaik berdasarkan nilai **MAE (Mean Absolute Error)** pada validation set.

---

## Konsep Dasar

| Istilah | Penjelasan |
|---|---|
| **Target (`y`)** | Nilai yang ingin diprediksi (misal: harga rumah) |
| **Fitur (`X`)** | Kolom input yang digunakan model untuk belajar |
| **Training set** | Data yang digunakan untuk melatih model |
| **Validation set** | Data yang digunakan untuk mengevaluasi performa model |
| **MAE** | Rata-rata selisih absolut antara prediksi dan nilai nyata |
| **Overfitting** | Model terlalu hafal data training, buruk di data baru |
| **Underfitting** | Model terlalu sederhana, tidak menangkap pola data |

---

## Alur Kerja

```
Load Data
    │
    ▼
Pilih Target (y) & Fitur (X)
    │
    ▼
Split: Training Set & Validation Set
    │
    ▼
Uji max_leaf_nodes 2–199 → hitung MAE tiap nilai
    │
    ▼
Pilih best_size (MAE terkecil)
    │
    ▼
Latih Model Final dengan seluruh data
    │
    ▼
Evaluasi & Prediksi
```

---

## Import Library

In [ ]:
import pandas as pd
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split

## Load & Bersihkan Data

Dataset yang digunakan adalah **Melbourne Housing Dataset** yang berisi data penjualan properti di Melbourne, Australia.

- `pd.read_csv()` — membaca file CSV menjadi DataFrame
- `dropna(axis=0)` — menghapus **baris** yang memiliki setidaknya satu nilai `NaN`

> **Mengapa `dropna`?** `DecisionTreeRegressor` tidak bisa menangani missing values secara langsung, sehingga baris yang tidak lengkap harus dihapus terlebih dahulu.

In [ ]:
melbourne_file_path = r"D:\file baru\mendata\Intro to Machine Learning\melb_data.csv"
melbourne_data = pd.read_csv(melbourne_file_path)
melbourne_data = melbourne_data.dropna(axis=0)

print(f"Jumlah baris setelah dropna: {len(melbourne_data)}")
melbourne_data.head()

## Tentukan Target (y) dan Fitur (X)

- **Target `y`**: kolom `Price` — nilai yang ingin diprediksi oleh model
- **Fitur `X`**: 4 kolom yang dipilih sebagai input model:

| Fitur | Keterangan |
|---|---|
| `Rooms` | Jumlah kamar |
| `Landsize` | Luas tanah (m²) |
| `Lattitude` | Koordinat lintang lokasi properti |
| `Longtitude` | Koordinat bujur lokasi properti |

> Pemilihan fitur yang relevan sangat mempengaruhi performa model. Fitur dengan korelasi kuat terhadap target akan menghasilkan prediksi yang lebih baik.

In [ ]:
y = melbourne_data.Price
melbourne_features = ['Rooms', 'Landsize', 'Lattitude', 'Longtitude']
X = melbourne_data[melbourne_features]

print("Statistik fitur X:")
print(X.describe())

## Split: Training Set & Validation Set

`train_test_split` membagi data secara acak menjadi dua bagian:

| Set | Proporsi Default | Digunakan untuk |
|---|---|---|
| **Training set** | 75% | Melatih model |
| **Validation set** | 25% | Mengevaluasi performa model |

- `random_state=1` — memastikan pembagian data selalu sama setiap kali kode dijalankan (reproducible)

> **Mengapa perlu validation set?** Jika kita mengevaluasi model pada data training yang sama, model hanya akan terlihat bagus karena ia sudah "hafal" data tersebut. Validation set mensimulasikan data baru yang belum pernah dilihat model.

In [ ]:
train_X, val_X, train_y, val_y = train_test_split(X, y, random_state=1)

print(f"Ukuran training set : {train_X.shape[0]} baris")
print(f"Ukuran validation set: {val_X.shape[0]} baris")

## Fungsi `get_mae` — Evaluasi Model per Ukuran Pohon

### Apa itu `max_leaf_nodes`?

Parameter ini mengontrol **seberapa dalam** pohon keputusan tumbuh:

```
max_leaf_nodes kecil  →  pohon dangkal  →  Underfitting  →  MAE tinggi
max_leaf_nodes besar  →  pohon dalam   →  Overfitting   →  MAE tinggi di data baru
max_leaf_nodes optimal →  keseimbangan  →  MAE terkecil di validation set
```

### Cara kerja fungsi:
1. Buat model `DecisionTreeRegressor` dengan `max_leaf_nodes` tertentu
2. Latih model dengan `train_X` dan `train_y`
3. Prediksi `val_X`
4. Hitung dan kembalikan **MAE** antara prediksi dan `val_y`

In [ ]:
def get_mae(max_leaf_nodes, train_X, val_X, train_y, val_y):
    model = DecisionTreeRegressor(max_leaf_nodes=max_leaf_nodes, random_state=1)
    model.fit(train_X, train_y)
    val_preds = model.predict(val_X)
    mae = mean_absolute_error(val_y, val_preds)
    return mae

## Cari `max_leaf_nodes` Terbaik

Uji semua nilai `max_leaf_nodes` dari **2 hingga 199** menggunakan *dictionary comprehension*, lalu pilih nilai dengan **MAE terkecil**.

```python
scores = {max_leaf_nodes: get_mae(...) for max_leaf_nodes in range(2, 200)}
```

Hasilnya adalah dictionary:
```
{ 2: 368737.xx, 3: 309813.xx, ..., 50: 257690.xx, ..., 199: 274012.xx }
```

`min(scores, key=scores.get)` → ambil *key* (nilai `max_leaf_nodes`) yang memiliki *value* (MAE) terkecil.

In [ ]:
scores = {
    max_leaf_nodes: get_mae(max_leaf_nodes, train_X, val_X, train_y, val_y)
    for max_leaf_nodes in range(2, 200)
}

best_size = min(scores, key=scores.get)
print(f"Model dengan nodes terbaik: {best_size} (MAE validasi: {scores[best_size]:,.2f})")

## Latih Model Final

Setelah menemukan `best_size`, latih model final menggunakan **seluruh data (`X`, `y`)** — bukan hanya training set.

> **Mengapa melatih ulang dengan seluruh data?**  
> Semakin banyak data yang dilihat model saat training, semakin baik pola yang bisa ia pelajari. Karena kita sudah menentukan hyperparameter (`best_size`) menggunakan validation set, sekarang data validation bisa ikut digunakan untuk melatih model final.

Kemudian evaluasi model final pada `val_X` untuk melihat MAE akhirnya.

In [ ]:
# Latih model final dengan seluruh data
final_model = DecisionTreeRegressor(max_leaf_nodes=best_size, random_state=1)
final_model.fit(X, y)

# Evaluasi pada validation set
final_val_preds = final_model.predict(val_X)
final_mae = mean_absolute_error(val_y, final_val_preds)

print(f"Prediksi 5 data pertama validation set:")
print(final_val_preds[:5].round(2))
print(f"\nHarga asli 5 data pertama validation set:")
print(val_y.values[:5])
print(f"\nMAE model final pada validation set: {final_mae:,.2f}")

---

## Rangkuman

| Langkah | Kode Utama | Tujuan |
|---|---|---|
| Load data | `pd.read_csv()` + `dropna()` | Memuat dan membersihkan dataset |
| Pilih target & fitur | `y = ...Price`, `X = ...[features]` | Menentukan input dan output model |
| Split data | `train_test_split()` | Memisahkan data training dan validasi |
| Cari hyperparameter | `get_mae()` + loop | Menemukan `max_leaf_nodes` optimal |
| Latih model final | `final_model.fit(X, y)` | Latih dengan seluruh data |
| Evaluasi | `mean_absolute_error()` | Mengukur akurasi model |

### Interpretasi MAE

MAE sebesar **X** berarti rata-rata prediksi harga model meleset sebesar **X dollar** dari harga aslinya.
Semakin kecil MAE, semakin akurat model.

### Overfitting vs Underfitting

```
         MAE
          ▲
    tinggi│\                    /
          │ \  Underfitting    / Overfitting
          │  \               /
     rendah│   \_____________/
          └──────────────────────► max_leaf_nodes
               kecil    optimal    besar
```

Titik optimal berada di lembah kurva — nilai `max_leaf_nodes` yang menghasilkan MAE terkecil pada validation set.